
# Panda 100k Evaluation -- Clean Consolidated Rebuild

**This notebook replaces the fragmented gate -> diagnostic -> 50k-recompute
sequence from the previous session with one clean, self-contained flow.**

**Important epistemic note, stated explicitly rather than left implicit:**
the original design called for a blind, pre-registered in-distribution gate
(decide the threshold, then look at the number). That is no longer possible
for this specific checkpoint pair: baseline_100k's Lorenz MAE has already
been observed across multiple protocol/horizon combinations in the prior
session. This notebook is therefore **exploratory re-analysis**, not a
confirmatory gate -- it presents a full table across all reasonable
protocol/horizon choices rather than a single automatically-computed
pass/fail verdict, and the decision about how to read that table is made
explicitly by inspection. Any future checkpoint pair (e.g. after further
training) should use a freshly pre-registered gate, informed by whatever
protocol/horizon this table suggests is most appropriate.

**Structure:**
1. Setup
2. Robust checkpoint locator (searches by content, not assumed path depth)
3. Load all four checkpoints: baseline_50k, ablation_50k, baseline_100k, ablation_100k
4. Harness (generalized to take pipeline objects as arguments, not globals)
5. Three Lorenz protocols, isolating channel count from IC/integrator
6. Optional held-out skew40 systems (Lorenz-only is a valid fallback)
7. Master diagnostic: all four checkpoints x three protocols x two horizons, one table
8. OOD loaders (Weather, Burgers, Van der Pol, Duffing, Harmonic)
9. OOD evaluation -- manually gated on reviewing the master diagnostic first

**Before running:** adjust `DATASET_100K_HINT`, `DATASET_50K_HINT`, `DATA_DIR`,
and the `sys.path.insert` line to match your actual Kaggle session paths.


## 1. Setup

In [2]:
import os, json, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 20
CONTEXT_LEN = 512
OUT_NPZ_DIR = './eval100k_raw_predictions'
os.makedirs(OUT_NPZ_DIR, exist_ok=True)


Device: cuda


## 3. Load All Four Checkpoints

**ADJUST** `DATASET_100K_HINT` and `DATASET_50K_HINT` to your actual dataset slugs before running.

In [6]:
import sys
sys.path.insert(0, '/kaggle/working/panda')  # ADJUST if your session clones panda elsewhere
from panda.patchtst.pipeline import PatchTSTPipeline


CHECKPOINT_DIRS = {
    'baseline_100k': '/kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-baseline-checkpoint/panda-100k-baseline-checkpoint',
    'ablation_100k': '/kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-ablation-checkpoint/panda-100k-ablation-checkpoint',
    'baseline_50k':  '/kaggle/input/datasets/anujb2/panda-50k-checkpoints/baseline/baseline/checkpoint-final',
    'ablation_50k':  '/kaggle/input/datasets/anujb2/panda-50k-checkpoints/koopman_ablation/koopman_ablation/checkpoint-final',
}

PIPES = {}
for name, ckpt_dir in CHECKPOINT_DIRS.items():
    with open(os.path.join(ckpt_dir, 'training_info.json')) as f:
        info = json.load(f)
    steps = info.get('total_steps', info.get('max_steps'))
    print(f'{name}: dir={ckpt_dir}')
    print(f'         steps={steps}, use_dynamics_embedding={info.get("use_dynamics_embedding")}')
    PIPES[name] = PatchTSTPipeline.from_pretrained(
        mode='predict', pretrain_path=ckpt_dir, device_map=device,
    )

print('\nAll four checkpoints loaded: baseline_100k, ablation_100k, baseline_50k, ablation_50k.')
print('Access via PIPES[name].')


baseline_100k: dir=/kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-baseline-checkpoint/panda-100k-baseline-checkpoint
         steps=100000, use_dynamics_embedding=True
ablation_100k: dir=/kaggle/input/datasets/anujb2/eval-checkpoints/panda-100k-ablation-checkpoint/panda-100k-ablation-checkpoint
         steps=100000, use_dynamics_embedding=False
baseline_50k: dir=/kaggle/input/datasets/anujb2/panda-50k-checkpoints/baseline/baseline/checkpoint-final
         steps=50000, use_dynamics_embedding=True
ablation_50k: dir=/kaggle/input/datasets/anujb2/panda-50k-checkpoints/koopman_ablation/koopman_ablation/checkpoint-final
         steps=50000, use_dynamics_embedding=False

All four checkpoints loaded: baseline_100k, ablation_100k, baseline_50k, ablation_50k.
Access via PIPES[name].


## 4. Harness

Generalized versions of `panda_forecast`, `single_condition_mae`, and `paired_evaluate` -- all take pipeline objects as arguments rather than closing over hardcoded globals, so the same functions work for any pair (50k vs 100k, baseline vs ablation, etc.) without duplication.

In [7]:
def instance_norm_window(x_CT):
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def panda_forecast_with(pipe, context_np, horizon):
    # Verbatim panda_forecast logic, parametrised by pipeline object.
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = pipe.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)


def single_condition_mae(pipe, data_CT, horizon, n_windows=N_WINDOWS):
    # Standalone MAE for one model on one dataset -- the building block
    # for both the diagnostic table and any convergence comparison.
    C, T = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts = np.linspace(0, max_start, n_windows, dtype=int)
    maes = []
    for s in starts:
        ctx_raw = data_CT[:, s:s+CONTEXT_LEN]
        tgt_raw = data_CT[:, s+CONTEXT_LEN:s+CONTEXT_LEN+horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std
        p = panda_forecast_with(pipe, ctx_norm, horizon)
        maes.append(mae(tgt_norm, p))
    return float(np.median(maes))


def paired_evaluate(pipe_a, pipe_b, data_CT, horizon, label,
                     name_a='a', name_b='b', n_windows=N_WINDOWS, save_npz=True):
    # Direct paired comparison of two named pipelines on the SAME
    # windows. Generalized (pipe_a/pipe_b as arguments) so it works for
    # any pair -- 100k baseline vs ablation, or 50k vs 100k of the same
    # arm, etc. -- rather than being hardcoded to two global pipelines.
    C, T = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []
    preds_a, preds_b, tgts = [], [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        pa = panda_forecast_with(pipe_a, ctx_norm, horizon)
        pb = panda_forecast_with(pipe_b, ctx_norm, horizon)
        mae_a.append(mae(tgt_norm, pa))
        mae_b.append(mae(tgt_norm, pb))
        preds_a.append(pa); preds_b.append(pb); tgts.append(tgt_norm)

    if save_npz:
        np.savez_compressed(
            f'{OUT_NPZ_DIR}/{label}.npz',
            preds_a=np.array(preds_a), preds_b=np.array(preds_b),
            targets=np.array(tgts), starts=starts,
        )

    diff = np.array(mae_b) - np.array(mae_a)  # >0 means b worse than a
    try:
        _, p_b_worse  = wilcoxon(diff, alternative='greater') if np.any(diff != 0) else (0, 1.0)
        _, p_b_better = wilcoxon(diff, alternative='less')    if np.any(diff != 0) else (0, 1.0)
    except Exception:
        p_b_worse = p_b_better = np.nan

    result = {
        'label': label, 'horizon': horizon, 'n_windows': n_windows,
        f'{name_a}_mae': np.median(mae_a),
        f'{name_a}_iqr': np.percentile(mae_a,75) - np.percentile(mae_a,25),
        f'{name_b}_mae': np.median(mae_b),
        f'{name_b}_iqr': np.percentile(mae_b,75) - np.percentile(mae_b,25),
        f'{name_b}_minus_{name_a}': np.median(mae_b) - np.median(mae_a),
        f'p_{name_b}_worse':  p_b_worse,
        f'p_{name_b}_better': p_b_better,
    }
    sig = f' *{name_b.upper()} WORSE' if p_b_worse < 0.05 else \
          (f' *{name_b.upper()} BETTER' if p_b_better < 0.05 else '')
    print(f'  {label:32s} H={horizon:4d}  {name_a}={np.median(mae_a):.4f}  '
          f'{name_b}={np.median(mae_b):.4f}  '
          f'\u0394={result[f"{name_b}_minus_{name_a}"]:+.4f}  '
          f'p(worse)={p_b_worse:.3f} p(better)={p_b_better:.3f}{sig}')
    return result


print('Harness defined: instance_norm_window, mae, panda_forecast_with, '
      'single_condition_mae, paired_evaluate.')


Harness defined: instance_norm_window, mae, panda_forecast_with, single_condition_mae, paired_evaluate.


## 5. Three Lorenz Protocols

- `gate_3ch`: fixed IC, manual RK4, 3 channels
- `gate_1ch`: same trajectory as gate_3ch, x-component only -- isolates channel count
- `alt_1ch`: seeded IC, solve_ivp/RK45, x-component only -- isolates IC+integrator (both 1ch, so compare directly against gate_1ch)

In [8]:
# =====================================================================
# Three Lorenz trajectories, isolating one variable at a time:
#   gate_3ch: fixed IC, manual RK4, 3 channels (x,y,z)
#   gate_1ch: SAME trajectory as gate_3ch, x-component only
#             -- isolates channel count alone
#   alt_1ch:  seeded IC, solve_ivp/RK45, x-component only (the protocol
#             used throughout new_experiments.ipynb) -- isolates
#             IC + integrator, independent of channel count (both 1ch)
# =====================================================================

def simulate_lorenz_gate(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3):
    # Fixed initial condition, manual RK4. Returns (T, 3).
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n - 1):
        k1x = sigma * (y - x); k1y = x * (rho - z) - y; k1z = x * y - beta * z
        k2x = sigma * ((y + dt/2*k1y) - (x + dt/2*k1x))
        k2y = (x + dt/2*k1x) * (rho - (z + dt/2*k1z)) - (y + dt/2*k1y)
        k2z = (x + dt/2*k1x) * (y + dt/2*k1y) - beta * (z + dt/2*k1z)
        k3x = sigma * ((y + dt/2*k2y) - (x + dt/2*k2x))
        k3y = (x + dt/2*k2x) * (rho - (z + dt/2*k2z)) - (y + dt/2*k2y)
        k3z = (x + dt/2*k2x) * (y + dt/2*k2y) - beta * (z + dt/2*k2z)
        k4x = sigma * ((y + dt*k3y) - (x + dt*k3x))
        k4y = (x + dt*k3x) * (rho - (z + dt*k3z)) - (y + dt*k3y)
        k4z = (x + dt*k3x) * (y + dt*k3y) - beta * (z + dt*k3z)
        x += dt/6*(k1x+2*k2x+2*k3x+k4x)
        y += dt/6*(k1y+2*k2y+2*k3y+k4y)
        z += dt/6*(k1z+2*k2z+2*k3z+k4z)
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs]).T


def alt_simulate_lorenz(n_steps=6000, dt=0.01, rho=28.0, seed=SEED):
    # Seeded random IC, solve_ivp/RK45, x-component only. This is the
    # protocol used throughout new_experiments.ipynb (Experiments 3,
    # 14, 19) -- the only Lorenz simulator that existed prior to the
    # eval notebook, so plausibly (not certainly) what produced any
    # historical reference figures.
    rng = np.random.default_rng(seed)
    def lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
        x, y, z = state
        return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]
    ic = rng.standard_normal(3)
    t_span = (0, n_steps * dt)
    t_eval = np.linspace(*t_span, n_steps)
    sol = solve_ivp(lorenz_rhs, t_span, ic, t_eval=t_eval,
                     args=(10.0, rho, 8.0/3.0),
                     method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y[0].astype(np.float32)


lorenz_traj_gate = simulate_lorenz_gate(n=5000)[500:3500]  # discard transient
lorenz_gate_3ch  = lorenz_traj_gate.T                       # (3, 3000)
lorenz_gate_1ch  = lorenz_gate_3ch[0:1, :]                  # (1, 3000) -- same traj, x only
lorenz_alt_1ch   = alt_simulate_lorenz(n_steps=6000, seed=SEED)[500:][None, :]  # (1, 5500)

PROTOCOLS = {
    'gate_3ch (fixed IC, manual RK4, 3ch)': lorenz_gate_3ch,
    'gate_1ch (fixed IC, manual RK4, 1ch)': lorenz_gate_1ch,
    'alt_1ch  (seeded IC, RK45, 1ch)':      lorenz_alt_1ch,
}

for name, data_CT in PROTOCOLS.items():
    print(f'{name}: shape={data_CT.shape}')

HORIZONS = [96, 336]  # 96 = single-pass (no rollout); 336 = 3 chained rollout passes


gate_3ch (fixed IC, manual RK4, 3ch): shape=(3, 3000)
gate_1ch (fixed IC, manual RK4, 1ch): shape=(1, 3000)
alt_1ch  (seeded IC, RK45, 1ch): shape=(1, 5500)


## 6. Optional: Held-Out skew40 Systems

Strengthens the in-distribution read beyond Lorenz alone if available. Lorenz-only is a valid, weaker fallback -- does not block anything below.

In [11]:
from datasets import load_dataset
hf_dataset = load_dataset('GilpinLab/skew40', split='train')

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/2.15G [00:00<?, ?B/s]

test_zeroshot.parquet:   0%|          | 0.00/857M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20979 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8250 [00:00<?, ? examples/s]

In [12]:
# Optional: two skew40 systems confirmed ABSENT from training, to
# strengthen the in-distribution read beyond Lorenz alone. Not required
# to proceed -- Lorenz-only is a weaker but functional fallback.
held_out_trajectories = {
    # 'system_name_1': traj_CT_1,   # TODO: fill in if available
    # 'system_name_2': traj_CT_2,   # TODO: fill in if available
}

if not held_out_trajectories:
    print('held_out_trajectories is empty -- proceeding on Lorenz alone. '
          'This weakens the in-distribution read but does not block it.')

# Optional helper: lists unique source systems in the loaded skew40
# corpus, if hf_dataset is in scope, so you can identify held-out
# candidates later.
try:
    unique_sources = sorted(set(hf_dataset['_source_directory']))
    print(f'\n{len(unique_sources)} unique source systems in the loaded skew40 split:')
    for s in unique_sources:
        print(f'  {s}')
except NameError:
    pass


held_out_trajectories is empty -- proceeding on Lorenz alone. This weakens the in-distribution read but does not block it.

1150 unique source systems in the loaded skew40 split:
  Aizawa_ForcedVanDerPol
  Aizawa_HyperLu
  Aizawa_NewtonLiepnik
  Aizawa_NuclearQuadrupole
  AnishchenkoAstakhov_BlinkingVortex
  AnishchenkoAstakhov_GuckenheimerHolmes
  AnishchenkoAstakhov_HyperCai
  AnishchenkoAstakhov_HyperLorenz
  AnishchenkoAstakhov_HyperYan
  AnishchenkoAstakhov_InteriorSquirmer
  AnishchenkoAstakhov_LorenzCoupled
  AnishchenkoAstakhov_SanUmSrisuchinwong
  AnishchenkoAstakhov_ShimizuMorioka
  AnishchenkoAstakhov_SprottM
  AnishchenkoAstakhov_Thomas
  AnishchenkoAstakhov_WangSun
  Arneodo_ArnoldBeltramiChildress
  Arneodo_Bouali2
  Arneodo_CaTwoPlusQuasiperiodic
  Arneodo_Chen
  Arneodo_Finance
  Arneodo_Hadley
  Arneodo_HenonHeiles
  Arneodo_LorenzBounded
  Arneodo_LuChenCheng
  Arneodo_NoseHoover
  Arneodo_SprottM
  Arneodo_SwingingAtwood
  Arneodo_Tsucs2
  Arneodo_WindmiReduced
  Arn

In [13]:
def simulate_rossler(n_steps=3000, a=0.2, b=0.2, c=5.7, seed=SEED):
    # Verbatim from new_experiments.ipynb (Cell 40).
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
    ic  = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps) -- x, y, z all returned


def simulate_sprott_b(n_steps=3000, seed=SEED):
    # Sprott System B (Sprott, 1994, "Some simple chaotic flows"):
    # xdot = yz, ydot = x - y, zdot = 1 - xy.
    rng = np.random.default_rng(seed)
    def rhs(t, state):
        x, y, z = state
        return [y*z, x - y, 1 - x*y]
    ic  = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y  # (3, n_steps)


rossler_traj  = simulate_rossler(n_steps=4000, seed=SEED)[:, 500:]   # (3, 3500)
sprottb_traj  = simulate_sprott_b(n_steps=4000, seed=SEED)[:, 500:]  # (3, 3500)

print(f'Rossler trajectory: {rossler_traj.shape}')
print(f'SprottB trajectory: {sprottb_traj.shape}')

held_out_trajectories = {
    'Rossler': rossler_traj,
    'SprottB': sprottb_traj,
}

print('\nheld_out_trajectories filled with Rossler and SprottB.')

Rossler trajectory: (3, 3500)
SprottB trajectory: (3, 3500)

held_out_trajectories filled with Rossler and SprottB.


In [14]:
# =====================================================================
# Extends the master diagnostic (Section 7) to include the held-out
# systems (Rossler, SprottB) -- genuinely different dynamics, not just
# protocol variants of Lorenz. This is the more informative test of
# whether the ablation-beats-baseline pattern reflects something about
# generalization broadly, or is specific to Lorenz.
#
# Uses gate_3ch-style evaluation (3 channels, matching held_out_trajectories'
# shape) for consistency -- the channel-count discussion earlier favored
# 3-channel as closer to the training corpus's own channel-count floor.
# =====================================================================

held_out_rows = []
print(f'{"system":12s} {"H":>4} {"base_50k":>10} {"base_100k":>10} {"abl_50k":>10} {"abl_100k":>10}')
print('-' * 65)

for sys_name, data_CT in held_out_trajectories.items():
    for h in HORIZONS:
        b50  = single_condition_mae(PIPES['baseline_50k'],  data_CT, h)
        a50  = single_condition_mae(PIPES['ablation_50k'],  data_CT, h)
        b100 = single_condition_mae(PIPES['baseline_100k'], data_CT, h)
        a100 = single_condition_mae(PIPES['ablation_100k'], data_CT, h)

        held_out_rows.append({
            'system': sys_name, 'horizon': h,
            'baseline_50k': b50, 'baseline_100k': b100,
            'baseline_improved': b100 < b50,
            'ablation_50k': a50, 'ablation_100k': a100,
            'ablation_improved': a100 < a50,
            'ablation_beats_baseline_100k': a100 < b100,
        })
        print(f'{sys_name:12s} {h:>4} {b50:>10.4f} {b100:>10.4f} {a50:>10.4f} {a100:>10.4f}')

df_held_out = pd.DataFrame(held_out_rows)
df_held_out.to_csv('held_out_systems_diagnostic.csv', index=False)
print('\nSaved held_out_systems_diagnostic.csv\n')

print(df_held_out[['system', 'horizon', 'ablation_beats_baseline_100k']].to_string(index=False))

n_lorenz_rows = df_master[df_master.protocol.str.startswith('gate_3ch')].shape[0]
n_lorenz_abl_wins = (df_master[df_master.protocol.str.startswith('gate_3ch')]['ablation_100k']
                     < df_master[df_master.protocol.str.startswith('gate_3ch')]['baseline_100k']).sum()
n_held_out_abl_wins = df_held_out['ablation_beats_baseline_100k'].sum()

print(f'\nLorenz (gate_3ch protocol): ablation beat baseline in '
      f'{n_lorenz_abl_wins}/{n_lorenz_rows} horizon combinations.')
print(f'Held-out systems (Rossler, SprottB): ablation beat baseline in '
      f'{n_held_out_abl_wins}/{len(df_held_out)} system/horizon combinations.')
print('\nIf both rates are similarly high, the pattern generalizes beyond Lorenz.')
print('If the held-out rate is much lower, the Lorenz result may be system-specific.')

system          H   base_50k  base_100k    abl_50k   abl_100k
-----------------------------------------------------------------
Rossler        96     0.5153     0.4765     0.2524     0.2512
Rossler       336     0.6171     0.6307     0.3703     0.3876
SprottB        96     0.3932     0.4132     0.2955     0.3098
SprottB       336     0.8014     0.7968     0.7826     0.7516

Saved held_out_systems_diagnostic.csv

 system  horizon  ablation_beats_baseline_100k
Rossler       96                          True
Rossler      336                          True
SprottB       96                          True
SprottB      336                          True

Lorenz (gate_3ch protocol): ablation beat baseline in 2/2 horizon combinations.
Held-out systems (Rossler, SprottB): ablation beat baseline in 4/4 system/horizon combinations.

If both rates are similarly high, the pattern generalizes beyond Lorenz.
If the held-out rate is much lower, the Lorenz result may be system-specific.


In [15]:
# =====================================================================
# Direct paired Wilcoxon test, baseline_100k vs ablation_100k, across
# every system now available: the three Lorenz protocols (gate_3ch,
# gate_1ch, alt_1ch) plus the two held-out systems (Rossler, SprottB).
# This converts the descriptive pattern above into an actually-tested
# claim -- the gap this whole diagnostic sequence has had since the
# start.
# =====================================================================

ALL_SYSTEMS = dict(PROTOCOLS)
ALL_SYSTEMS.update(held_out_trajectories)

sig_rows = []
print(f'{"system":40s} {"H":>4} {"baseline":>10} {"ablation":>10} {"delta":>9} '
      f'{"p(worse)":>9} {"p(better)":>10}')
print('-' * 100)

for sys_name, data_CT in ALL_SYSTEMS.items():
    for h in HORIZONS:
        r = paired_evaluate(
            PIPES['baseline_100k'], PIPES['ablation_100k'],
            data_CT, h, label=f'{sys_name}_H{h}',
            name_a='baseline', name_b='ablation',
            save_npz=True,
        )
        if r:
            r['system'] = sys_name
            sig_rows.append(r)

df_sig = pd.DataFrame(sig_rows)
df_sig.to_csv('baseline_vs_ablation_significance.csv', index=False)
print('\nSaved baseline_vs_ablation_significance.csv\n')

n_sig_ablation_better = (df_sig['p_ablation_better'] < 0.05).sum()
n_total = len(df_sig)
print(f'Ablation significantly better than baseline (p<0.05) in '
      f'{n_sig_ablation_better}/{n_total} system/horizon combinations.')

print('\n=== Full results ===')
print(df_sig[['system', 'horizon', 'baseline_mae', 'ablation_mae',
              'ablation_minus_baseline', 'p_ablation_worse',
              'p_ablation_better']].to_string(index=False))

system                                      H   baseline   ablation     delta  p(worse)  p(better)
----------------------------------------------------------------------------------------------------
  gate_3ch (fixed IC, manual RK4, 3ch)_H96 H=  96  baseline=0.6951  ablation=0.3462  Δ=-0.3489  p(worse)=0.999 p(better)=0.001 *ABLATION BETTER
  gate_3ch (fixed IC, manual RK4, 3ch)_H336 H= 336  baseline=0.9679  ablation=0.8097  Δ=-0.1582  p(worse)=0.923 p(better)=0.082
  gate_1ch (fixed IC, manual RK4, 1ch)_H96 H=  96  baseline=0.7515  ablation=0.5907  Δ=-0.1607  p(worse)=0.997 p(better)=0.004 *ABLATION BETTER
  gate_1ch (fixed IC, manual RK4, 1ch)_H336 H= 336  baseline=0.9587  ablation=0.8992  Δ=-0.0595  p(worse)=0.522 p(better)=0.493
  alt_1ch  (seeded IC, RK45, 1ch)_H96 H=  96  baseline=0.6786  ablation=0.4832  Δ=-0.1954  p(worse)=0.923 p(better)=0.082
  alt_1ch  (seeded IC, RK45, 1ch)_H336 H= 336  baseline=0.9390  ablation=0.7909  Δ=-0.1480  p(worse)=0.959 p(better)=0.045 *ABLATION B

## 7. Master Diagnostic

**The core deliverable.** All four checkpoints, all three protocols, both horizons, in one consolidated table. Read this table yourself before deciding anything -- no automatic verdict is computed.

In [10]:
# =====================================================================
# MASTER DIAGNOSTIC: single_condition_mae for all four checkpoints
# (baseline_50k, ablation_50k, baseline_100k, ablation_100k) across all
# three Lorenz protocols and both horizons. One consolidated table,
# computed in one pass, replacing the fragmented gate -> diagnostic ->
# 50k-recompute sequence from before.
#
# EPISTEMIC NOTE, stated explicitly rather than left implicit: because
# baseline_100k's numbers on several of these protocol/horizon
# combinations were already observed before this notebook was built,
# this table cannot honestly be presented as a blind, confirmatory,
# pre-registered gate. It is exploratory re-analysis. No single
# pass/fail verdict is computed automatically below -- all numbers are
# shown, and the decision about what they mean is made explicitly,
# by inspection, not by an opaque threshold that was partly fit after
# seeing results. Any FUTURE checkpoint pair (e.g. after further
# training) should use a freshly pre-registered gate decided before
# those numbers exist, informed by what protocol/horizon choice this
# table suggests is most appropriate going forward.
# =====================================================================

master_rows = []
print(f'{"protocol":40s} {"H":>4} {"base_50k":>10} {"base_100k":>10} {"abl_50k":>10} {"abl_100k":>10}')
print('-' * 100)

for proto_name, data_CT in PROTOCOLS.items():
    for h in HORIZONS:
        b50  = single_condition_mae(PIPES['baseline_50k'],  data_CT, h)
        a50  = single_condition_mae(PIPES['ablation_50k'],  data_CT, h)
        b100 = single_condition_mae(PIPES['baseline_100k'], data_CT, h)
        a100 = single_condition_mae(PIPES['ablation_100k'], data_CT, h)

        master_rows.append({
            'protocol': proto_name, 'horizon': h,
            'baseline_50k': b50, 'baseline_100k': b100,
            'baseline_improved': b100 < b50,
            'baseline_ratio_100k_over_50k': b100 / b50,
            'ablation_50k': a50, 'ablation_100k': a100,
            'ablation_improved': a100 < a50,
            'ablation_ratio_100k_over_50k': a100 / a50,
        })
        print(f'{proto_name:40s} {h:>4} {b50:>10.4f} {b100:>10.4f} {a50:>10.4f} {a100:>10.4f}')

df_master = pd.DataFrame(master_rows)
df_master.to_csv('master_diagnostic.csv', index=False)
print('\nSaved master_diagnostic.csv\n')

print('=== Improvement summary (100k vs matched-protocol 50k) ===')
print(df_master[['protocol', 'horizon', 'baseline_improved',
                  'baseline_ratio_100k_over_50k', 'ablation_improved',
                  'ablation_ratio_100k_over_50k']].to_string(index=False))

n_base_improved = df_master['baseline_improved'].sum()
n_abl_improved  = df_master['ablation_improved'].sum()
print(f'\nBaseline improved in {n_base_improved}/{len(df_master)} protocol/horizon combinations.')
print(f'Ablation improved in {n_abl_improved}/{len(df_master)} protocol/horizon combinations.')


protocol                                    H   base_50k  base_100k    abl_50k   abl_100k
----------------------------------------------------------------------------------------------------
gate_3ch (fixed IC, manual RK4, 3ch)       96     0.6195     0.6951     0.3880     0.3462
gate_3ch (fixed IC, manual RK4, 3ch)      336     0.9857     0.9679     0.8485     0.8097
gate_1ch (fixed IC, manual RK4, 1ch)       96     0.7410     0.7515     0.5736     0.5907
gate_1ch (fixed IC, manual RK4, 1ch)      336     1.0839     0.9587     1.0442     0.8992
alt_1ch  (seeded IC, RK45, 1ch)            96     0.6532     0.6786     0.5096     0.4832
alt_1ch  (seeded IC, RK45, 1ch)           336     0.8465     0.9390     0.8999     0.7909

Saved master_diagnostic.csv

=== Improvement summary (100k vs matched-protocol 50k) ===
                            protocol  horizon  baseline_improved  baseline_ratio_100k_over_50k  ablation_improved  ablation_ratio_100k_over_50k
gate_3ch (fixed IC, manual RK4, 3ch)

## 8. OOD Loaders

Reused verbatim from `new_experiments.ipynb` (Weather via `load_ts`, Burgers via `simulate_burgers_stable` + `pca_reduction`, Van der Pol / Duffing / Harmonic via their respective simulators).

In [18]:
DATA_DIR = '/kaggle/input/datasets/anujb2/eval-checkpoints/ts_data/ts_data'  # adjust if this differs from new_experiments.ipynb's path

def load_ts(path):
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)


def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=SEED):
    from scipy.fft import fft, ifft, fftfreq
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x

    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4    = rhs_hat(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f'    Diverged at t={t}')
                return U[:t]
    return U


def pca_reduction(U, n_components):
    from scipy.linalg import svd
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)


def simulate_harmonic(n_steps=3000, omega=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    return np.array(traj, dtype=np.float32)


def simulate_vanderpol(n_steps=3000, mu=2.0, seed=SEED):
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic  = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)


def simulate_duffing(n_steps=3000, delta=0.3, alpha=-1.0,
                     beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t    = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax    = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32)


def load_weather():
    return load_ts(f'{DATA_DIR}/weather.csv')

def load_burgers_nu1():
    U = simulate_burgers_stable(T=1500, N_x=128, nu=1.0, seed=SEED)
    pca_series = pca_reduction(U, 16)
    return pca_series.T

def load_vanderpol():
    series = simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED)
    return series[500:][None, :]

def load_duffing():
    series = simulate_duffing(n_steps=4000, seed=SEED)
    return series[500:][None, :]

def load_harmonic():
    series = simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED)
    return series[500:][None, :]

OOD_LOADERS = {
    'Weather':     (load_weather,     [96, 192, 336]),
    'Burgers_nu1': (load_burgers_nu1, [96, 192, 336]),
    'VanDerPol':   (load_vanderpol,   [96, 192, 336]),
    'Duffing':     (load_duffing,     [96, 192, 336]),
    'Harmonic':    (load_harmonic,    [96, 192, 336]),
}

print('OOD loaders defined: Weather, Burgers_nu1, VanDerPol, Duffing, Harmonic.')


OOD loaders defined: Weather, Burgers_nu1, VanDerPol, Duffing, Harmonic.


## 9. OOD Evaluation -- Manually Gated

Review the master diagnostic table from Section 7 first. Set `PROCEED_TO_OOD = True` explicitly once you've decided how to read it -- this is a judgment call given the mixed picture across protocols, not something to automate.

In [19]:
# =====================================================================
# STAGE 2 -- OOD evaluation of baseline_100k vs ablation_100k.
#
# Manually gated rather than automatically gated on a single threshold:
# given the master diagnostic above shows a genuinely mixed picture
# across protocols/horizons (review it yourself before proceeding),
# an automatic pass/fail here would hide exactly the judgment call that
# matters. Read the master diagnostic table, decide, then set the flag
# below explicitly.
# =====================================================================

PROCEED_TO_OOD = True  # <-- set to True only after reviewing master_diagnostic.csv

if PROCEED_TO_OOD:
    ood_results = []
    for name, (loader_fn, horizons) in OOD_LOADERS.items():
        try:
            data_CT = loader_fn()
        except NotImplementedError as e:
            print(f'[SKIP] {name}: {e}')
            continue
        print(f'\n=== {name} ===')
        for H in horizons:
            r = paired_evaluate(
                PIPES['baseline_100k'], PIPES['ablation_100k'],
                data_CT, H, label=f'{name}_H{H}',
                name_a='baseline', name_b='ablation',
            )
            if r:
                r['dataset'] = name
                ood_results.append(r)

    if ood_results:
        df_ood = pd.DataFrame(ood_results)
        df_ood.to_csv('ood_100k_results.csv', index=False)
        print('\nSaved ood_100k_results.csv')
        print(df_ood[['label', 'ablation_minus_baseline']].to_string(index=False))
    else:
        print('\nNo OOD datasets loaded.')
else:
    print('PROCEED_TO_OOD is False. Review master_diagnostic.csv, decide, '
          'then set PROCEED_TO_OOD = True and rerun this cell.')



=== Weather ===
  Weather_H96                      H=  96  baseline=0.7312  ablation=0.6942  Δ=-0.0370  p(worse)=0.608 p(better)=0.406
  Weather_H192                     H= 192  baseline=0.7983  ablation=0.8483  Δ=+0.0500  p(worse)=0.707 p(better)=0.293
  Weather_H336                     H= 336  baseline=0.9204  ablation=0.9984  Δ=+0.0780  p(worse)=0.063 p(better)=0.937

=== Burgers_nu1 ===
  Burgers_nu1_H96                  H=  96  baseline=0.0474  ablation=0.0686  Δ=+0.0212  p(worse)=0.011 p(better)=0.990 *ABLATION WORSE
  Burgers_nu1_H192                 H= 192  baseline=0.0764  ablation=0.0815  Δ=+0.0052  p(worse)=0.249 p(better)=0.763
  Burgers_nu1_H336                 H= 336  baseline=0.1666  ablation=0.1628  Δ=-0.0037  p(worse)=0.622 p(better)=0.392

=== VanDerPol ===
  VanDerPol_H96                    H=  96  baseline=0.1330  ablation=0.1167  Δ=-0.0163  p(worse)=0.934 p(better)=0.071
  VanDerPol_H192                   H= 192  baseline=0.1781  ablation=0.1856  Δ=+0.0076  p(wors